# CMT307 Applied Machine Learning — ASHRAE Great Energy Predictor III

**Cardiff University | Spring 2025/26 | Task 9: Energy Usage Prediction**

---

| | |
|---|---|
| **Name** | Zahra |
| **Role** | Person 4 — Data Merging & Integration Hub |
| **Notebook** | `Zahra_Data_Merging.ipynb` |

In [1]:
import pandas as pd
import os

DATA_DIR = '../data/ashrae-energy-prediction'

train        = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'), parse_dates=['timestamp'])
metadata     = pd.read_csv(os.path.join(DATA_DIR, 'building_metadata.csv'))
weather      = pd.read_csv(os.path.join(DATA_DIR, 'weather_train.csv'), parse_dates=['timestamp'])
test         = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'), parse_dates=['timestamp'])
weather_test = pd.read_csv(os.path.join(DATA_DIR, 'weather_test.csv'), parse_dates=['timestamp'])

print('train:       ', train.shape)
print('metadata:    ', metadata.shape)
print('weather:     ', weather.shape)
print('test:        ', test.shape)
print('weather_test:', weather_test.shape)

train:        (20216100, 4)
metadata:     (1449, 6)
weather:      (139773, 9)
test:         (41697600, 4)
weather_test: (277243, 9)


In [2]:
# Check all building_ids in train exist in metadata
missing_ids = set(train['building_id']) - set(metadata['building_id'])
print(f'building_ids in train not in metadata: {len(missing_ids)}')
print(f'unique buildings in train:    {train["building_id"].nunique()}')
print(f'unique buildings in metadata: {metadata["building_id"].nunique()}')

building_ids in train not in metadata: 0
unique buildings in train:    1449
unique buildings in metadata: 1449


In [3]:
# Merge train + metadata on building_id
merged_train = train.merge(metadata, on='building_id', how='left')
print('After train + metadata:', merged_train.shape)  # rows must stay at 20,216,100

# Merge + weather on site_id + timestamp
merged_train = merged_train.merge(weather, on=['site_id', 'timestamp'], how='left')
print('After + weather:       ', merged_train.shape)  # must still be 20,216,100

After train + metadata: (20216100, 9)
After + weather:        (20216100, 16)


In [4]:
# Null counts after merge
nulls = merged_train.isnull().sum()
print(nulls[nulls > 0])

year_built            12127645
floor_count           16709167
air_temperature          96658
cloud_coverage         8825365
dew_temperature         100140
precip_depth_1_hr      3749023
sea_level_pressure     1231669
wind_direction         1449048
wind_speed              143676
dtype: int64


In [5]:
# Repeat for test
merged_test = test.merge(metadata, on='building_id', how='left')
merged_test = merged_test.merge(weather_test, on=['site_id', 'timestamp'], how='left')
print('merged_test:', merged_test.shape)

nulls_test = merged_test.isnull().sum()
print(nulls_test[nulls_test > 0])

merged_test: (41697600, 16)
year_built            24598080
floor_count           34444320
air_temperature         221901
cloud_coverage        19542180
dew_temperature         260799
precip_depth_1_hr      7801563
sea_level_pressure     2516826
wind_direction         2978663
wind_speed              302089
dtype: int64


In [6]:
# Save merged files as CSV
os.makedirs('../data_processed', exist_ok=True)
merged_train.to_csv('../data_processed/merged_train.csv', index=False)
merged_test.to_csv('../data_processed/merged_test.csv', index=False)
print('Saved merged_train.csv and merged_test.csv to data_processed/')

Saved merged_train.csv and merged_test.csv to data_processed/


### Key Findings

- All 5 files loaded successfully: train (20,216,100 rows), metadata (1,449), weather_train (139,773), test (41,697,600), weather_test (277,243)
- All 1,449 building IDs in train are present in metadata — no orphan records
- Both merges preserved row count exactly (20,216,100 for train, 41,697,600 for test) — no row duplication or loss
- 16 columns in final merged tables (4 original + 5 metadata + 7 weather features)
- Columns with missing values after merge:
  - `year_built`: 60% missing — sparse in source metadata
  - `floor_count`: 83% missing — sparse in source metadata
  - `cloud_coverage`: 44% missing — weather station gaps
  - `precip_depth_1_hr`: 19% missing
  - `sea_level_pressure`: 6% missing
  - `wind_direction`: 7% missing
  - `air_temperature`, `dew_temperature`, `wind_speed`: <1% missing
- Missing weather values are due to gaps in the original weather station recordings, not a merge error
- Saved: `merged_train.csv` and `merged_test.csv` to `data_processed/`

---

## Sprint 2 — Data Merging, Preprocessing & Feature Engineering

In [2]:
import pandas as pd
import numpy as np
import os
import gc

PROCESSED_DIR = '../data_processed'
DATA_DIR      = '../data/ashrae-energy-prediction'
OUTPUTS_DIR   = '../outputs'

# Load Zahra's Sprint 1 merged files as the pipeline base
print("Loading Sprint 1 merged files...")
df_train = pd.read_csv(
    os.path.join(PROCESSED_DIR, 'merged_train.csv'),
    dtype={'building_id': 'int16', 'meter': 'int8', 'site_id': 'int8'},
    parse_dates=['timestamp']
)
df_test = pd.read_csv(
    os.path.join(PROCESSED_DIR, 'merged_test.csv'),
    dtype={'building_id': 'int16', 'meter': 'int8', 'site_id': 'int8'},
    parse_dates=['timestamp']
)

print(f"df_train : {df_train.shape}")
print(f"df_test  : {df_test.shape}")
print(f"Columns  : {list(df_train.columns)}")

Loading Sprint 1 merged files...
df_train : (20216100, 24)
df_test  : (41697600, 23)
Columns  : ['building_id', 'meter', 'timestamp', 'meter_reading', 'site_id', 'primary_use', 'square_feet', 'year_built', 'floor_count', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed', 'hour', 'day_of_week', 'month', 'day_of_year', 'season', 'is_weekend', 'is_business_hours', 'log_meter_reading']


In [3]:
# ── Card 8 (Wahid): Fix Missing Building Features + Encode Categories ─────────

# A: Flag missing floor_count BEFORE imputing (used as a model feature itself)
df_train['missing_floor_count_flag'] = df_train['floor_count'].isnull().astype(np.int8)
df_test['missing_floor_count_flag']  = df_test['floor_count'].isnull().astype(np.int8)

# B: Impute floor_count — group median cascade (primary_use + site_id + sqft_bin → fallbacks)
for df in [df_train, df_test]:
    df['sqft_bin'] = pd.qcut(df['square_feet'], q=5, duplicates='drop')
    df['floor_count'] = df['floor_count'].fillna(
        df.groupby(['primary_use', 'site_id', 'sqft_bin'], observed=True)['floor_count'].transform('median'))
    df['floor_count'] = df['floor_count'].fillna(
        df.groupby(['primary_use', 'sqft_bin'], observed=True)['floor_count'].transform('median'))
    df['floor_count'] = df['floor_count'].fillna(
        df.groupby('sqft_bin', observed=True)['floor_count'].transform('median'))
    df['floor_count'] = df['floor_count'].fillna(df['floor_count'].median())
    df['floor_count'] = df['floor_count'].round().astype(np.int8)
    df.drop(columns=['sqft_bin'], inplace=True)

# C: Impute year_built — group median cascade
for df in [df_train, df_test]:
    df['year_built'] = df['year_built'].fillna(
        df.groupby(['primary_use', 'site_id'], observed=True)['year_built'].transform('median'))
    df['year_built'] = df['year_built'].fillna(
        df.groupby('primary_use', observed=True)['year_built'].transform('median'))
    df['year_built'] = df['year_built'].fillna(df['year_built'].median())

# D: Derived features
for df in [df_train, df_test]:
    df['building_age']    = (2016 - df['year_built']).astype(np.int16)
    df['log_square_feet'] = np.log1p(df['square_feet']).astype(np.float32)

# E: One-hot encode primary_use (fit categories on train, apply same columns to test)
all_uses = sorted(df_train['primary_use'].unique())
for use in all_uses:
    col = f'use_{use}'
    df_train[col] = (df_train['primary_use'] == use).astype(np.int8)
    df_test[col]  = (df_test['primary_use']  == use).astype(np.int8)

# F: Drop raw columns replaced by engineered ones
for df in [df_train, df_test]:
    df.drop(columns=[c for c in ['primary_use', 'square_feet', 'year_built'] if c in df.columns], inplace=True)

gc.collect()
use_cols = [c for c in df_train.columns if c.startswith('use_')]
print(f"Building features applied | {len(use_cols)} use_* columns")
print(f"Null check — floor_count: {df_train['floor_count'].isnull().sum()} | building_age: {df_train['building_age'].isnull().sum()}")

Building features applied | 16 use_* columns
Null check — floor_count: 0 | building_age: 0


In [4]:
# ── Card 9 (Tanisha): Impute Weather + Engineer Weather Features ──────────────
# Load raw weather files and apply Tanisha's preprocessing at the station level
# (1 row per site+timestamp), then re-merge into the main dataframes

weather_train_raw = pd.read_csv(
    os.path.join(DATA_DIR, 'weather_train.csv'), parse_dates=['timestamp']
).sort_values(['site_id', 'timestamp']).reset_index(drop=True)

weather_test_raw = pd.read_csv(
    os.path.join(DATA_DIR, 'weather_test.csv'), parse_dates=['timestamp']
).sort_values(['site_id', 'timestamp']).reset_index(drop=True)

def preprocess_weather(df):
    df = df.copy()
    for col in ['air_temperature', 'dew_temperature']:
        df[col] = df.groupby('site_id')[col].transform(
            lambda x: x.interpolate(method='linear', limit_direction='both'))
    df['cloud_coverage'] = df.groupby('site_id')['cloud_coverage'].transform(
        lambda x: x.ffill().bfill())
    df['cloud_coverage'] = df.groupby('site_id')['cloud_coverage'].transform(
        lambda x: x.fillna(x.median()))
    df['cloud_coverage'] = df['cloud_coverage'].fillna(df['cloud_coverage'].median())
    df['precip_was_missing'] = df['precip_depth_1_hr'].isna().astype(np.int8)
    df['precip_depth_1_hr']  = df['precip_depth_1_hr'].fillna(0)
    for col in ['sea_level_pressure', 'wind_direction']:
        rolled = df.groupby('site_id')[col].transform(
            lambda x: x.rolling(24, min_periods=1).median())
        df[col] = df[col].fillna(rolled)
        df[col] = df.groupby('site_id')[col].transform(lambda x: x.fillna(x.median()))
    df['relative_humidity'] = (100 * (
        np.exp((17.625 * df['dew_temperature']) / (243.04 + df['dew_temperature'])) /
        np.exp((17.625 * df['air_temperature']) / (243.04 + df['air_temperature']))
    )).clip(0, 100)
    df['temp_diff_from_comfort'] = (df['air_temperature'] - 21).abs()
    return df

weather_train_proc = preprocess_weather(weather_train_raw)
weather_test_proc  = preprocess_weather(weather_test_raw)

# Cast float64 -> float32 before merge to halve memory usage
# (df_test has 41.7M rows -- float64 weather cols need 3.1 GB, float32 needs ~1.5 GB)
for wdf in [weather_train_proc, weather_test_proc]:
    for col in wdf.select_dtypes(include='float64').columns:
        wdf[col] = wdf[col].astype(np.float32)

gc.collect()

# Drop old raw weather columns and re-merge with processed ones
weather_raw_cols = ['air_temperature', 'cloud_coverage', 'dew_temperature',
                    'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed']
df_train.drop(columns=[c for c in weather_raw_cols if c in df_train.columns], inplace=True)
df_test.drop( columns=[c for c in weather_raw_cols if c in df_test.columns],  inplace=True)

df_train = df_train.merge(weather_train_proc, on=['site_id', 'timestamp'], how='left')
df_test  = df_test.merge( weather_test_proc,  on=['site_id', 'timestamp'], how='left')

del weather_train_proc, weather_test_proc, weather_train_raw, weather_test_raw
gc.collect()

# Final fallback: ~0.4% of rows have timestamps entirely absent from the weather files
weather_feature_cols = [
    'air_temperature', 'cloud_coverage', 'dew_temperature',
    'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed',
    'precip_was_missing', 'relative_humidity', 'temp_diff_from_comfort'
]
for df in [df_train, df_test]:
    for col in weather_feature_cols:
        if col in df.columns and df[col].isnull().any():
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.ffill().bfill())
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.fillna(x.median()))
            df[col] = df[col].fillna(df[col].median())

gc.collect()
remaining = {c: int(df_train[c].isnull().sum()) for c in weather_feature_cols if c in df_train.columns}
remaining = {k: v for k, v in remaining.items() if v > 0}
print(f"Weather features applied | df_train: {df_train.shape}")
print(f"Remaining nulls after fallback: {remaining if remaining else 'None -- all clean'}")

Weather features applied | df_train: (20216100, 43)
Remaining nulls after fallback: None -- all clean


In [5]:
# ── Card 7 (Shriya): Time Features + Target Transformation ───────────────────
def get_season(month):
    if month in [12, 1, 2]:  return 0  # winter
    elif month in [3, 4, 5]: return 1  # spring
    elif month in [6, 7, 8]: return 2  # summer
    else:                    return 3  # autumn

for df in [df_train, df_test]:
    df['hour']              = df['timestamp'].dt.hour.astype(np.int8)
    df['day_of_week']       = df['timestamp'].dt.dayofweek.astype(np.int8)
    df['month']             = df['timestamp'].dt.month.astype(np.int8)
    df['day_of_year']       = df['timestamp'].dt.dayofyear.astype(np.int16)
    df['season']            = df['month'].map(get_season).astype(np.int8)
    df['is_weekend']        = (df['day_of_week'] >= 5).astype(np.int8)
    df['is_business_hours'] = (
        (df['hour'] >= 8) & (df['hour'] <= 18) & (df['day_of_week'] < 5)
    ).astype(np.int8)

df_train['log_meter_reading'] = np.log1p(df_train['meter_reading'])
print("Time features applied:", ['hour','day_of_week','month','day_of_year','season','is_weekend','is_business_hours','log_meter_reading'])

# ── Card 6 (Ayan): Anomaly Cleaning ──────────────────────────────────────────
rows_start = len(df_train)

# Decision 1: Remove Site 0 electricity rows before 2016-05-20
# Ayan found 93.75% of these readings are exactly zero — a known calibration gap
site0_mask = (
    (df_train['site_id'] == 0) &
    (df_train['meter']   == 0) &
    (df_train['timestamp'] < '2016-05-20')
)
df_train = df_train[~site0_mask].copy()
gc.collect()
print(f"\nRemoved {rows_start - len(df_train):,} rows — Site 0 electricity calibration gap (before 2016-05-20)")

# Decision 2: Remove building/meter pairs flagged for zero streaks > 168 hours
anomaly_df = pd.read_csv(
    os.path.join(OUTPUTS_DIR, 'anomaly_report.csv'),
    usecols=['anomaly_type', 'building_id', 'meter']
)
zero_streak_pairs = (
    anomaly_df[anomaly_df['anomaly_type'] == 'zero_streak'][['building_id', 'meter']]
    .drop_duplicates()
)
print(f"Building/meter pairs with zero streaks >168 h: {len(zero_streak_pairs)}")

train_midx  = pd.MultiIndex.from_arrays([df_train['building_id'], df_train['meter']])
streak_midx = pd.MultiIndex.from_frame(zero_streak_pairs)
rows_before = len(df_train)
df_train = df_train[~train_midx.isin(streak_midx)].copy()
gc.collect()
print(f"Removed {rows_before - len(df_train):,} rows — zero streak building/meter pairs")
print(f"\nTotal removed: {rows_start - len(df_train):,} rows ({(rows_start - len(df_train))/rows_start*100:.2f}%)")
print(f"df_train after cleaning: {df_train.shape}")

Time features applied: ['hour', 'day_of_week', 'month', 'day_of_year', 'season', 'is_weekend', 'is_business_hours', 'log_meter_reading']

Removed 346,112 rows — Site 0 electricity calibration gap (before 2016-05-20)
Building/meter pairs with zero streaks >168 h: 313
Removed 2,380,183 rows — zero streak building/meter pairs

Total removed: 2,726,295 rows (13.49%)
df_train after cleaning: (17489805, 43)


In [6]:
# ── Final Null Check ─────────────────────────────────────────────────────────
skip_cols = {'timestamp', 'meter_reading', 'log_meter_reading', 'row_id'}
check_cols = [c for c in df_train.columns if c not in skip_cols]

nulls = df_train[check_cols].isnull().sum()
remaining = nulls[nulls > 0]
if len(remaining) == 0:
    print("No nulls in any feature column — pipeline is clean.")
else:
    print("WARNING — nulls remain:")
    print(remaining)

# ── Time-Based Train / Validation Split ──────────────────────────────────────
# Train: Jan–Oct 2016 | Val: Nov–Dec 2016
# Time-based split is essential — a random split would leak future timestamps into training
train_final = df_train[df_train['timestamp'] < '2016-11-01'].copy()
val_final   = df_train[df_train['timestamp'] >= '2016-11-01'].copy()

assert len(train_final) + len(val_final) == len(df_train), "Split lost rows!"

print(f"\ntrain_final : {train_final.shape}  {train_final['timestamp'].min().date()} → {train_final['timestamp'].max().date()}")
print(f"val_final   : {val_final.shape}  {val_final['timestamp'].min().date()} → {val_final['timestamp'].max().date()}")
print(f"Split       : Train {len(train_final)/len(df_train)*100:.1f}%  /  Val {len(val_final)/len(df_train)*100:.1f}%")

# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs(PROCESSED_DIR, exist_ok=True)
train_final.to_csv(os.path.join(PROCESSED_DIR, 'final_train.csv'), index=False)
val_final.to_csv(  os.path.join(PROCESSED_DIR, 'final_val.csv'),   index=False)
df_test.to_csv(    os.path.join(PROCESSED_DIR, 'final_test.csv'),  index=False)

print(f"\nSaved to {PROCESSED_DIR}/")
print(f"  final_train.csv — {train_final.shape}")
print(f"  final_val.csv   — {val_final.shape}")
print(f"  final_test.csv  — {df_test.shape}")

No nulls in any feature column — pipeline is clean.

train_final : (14529655, 43)  2016-01-01 → 2016-10-31
val_final   : (2960150, 43)  2016-11-01 → 2016-12-31
Split       : Train 83.1%  /  Val 16.9%

Saved to ../data_processed/
  final_train.csv — (14529655, 43)
  final_val.csv   — (2960150, 43)
  final_test.csv  — (41697600, 42)


### Sprint 2 Key Decisions

| Step | What Was Applied | Source |
|---|---|---|
| Building imputation | `floor_count`: group median cascade (primary_use + site_id + sqft_bin); `year_built`: same cascade | Wahid (Card 8) |
| Building features | `building_age` = 2016 − year_built; `log_square_feet` = log1p(square_feet); one-hot `primary_use` → 16 `use_*` columns | Wahid (Card 8) |
| Weather imputation | `air_temperature`, `dew_temperature`: linear interpolation per site; `cloud_coverage`: ffill/bfill + median; `precip`: fill 0 + flag; `sea_level_pressure`, `wind_direction`: 24h rolling median | Tanisha (Card 9) |
| Weather features | `relative_humidity`, `temp_diff_from_comfort`, `precip_was_missing` | Tanisha (Card 9) |
| Time features | `hour`, `day_of_week`, `month`, `day_of_year`, `season`, `is_weekend`, `is_business_hours`, `log_meter_reading` | Shriya (Card 7) |
| Anomaly removal | Site 0 electricity before 2016-05-20 removed; building/meter pairs with zero streaks >168 h removed | Ayan (Card 6) |
| Train/val split | Jan–Oct 2016 = train (~83%), Nov–Dec 2016 = val (~17%) — time-based to prevent leakage | Card 10 |

---

## Sprint 3 — Preprocessing Pipeline & Train/Val Split

---

## Sprint 4 — Evaluation & Report